# Connect 4 AI: Analysis Notebook

This notebook is part of my Connect 4 project (HTML, CSS, JavaScript, C++ and Python).
The game runs in the browser. This notebook compiles the **same AI in C++** and measures it.

**What it shows**
1. How many positions alpha-beta pruning saves compared with plain minimax
2. How fast the engine searches at each depth
3. A round-robin tournament: AIs with different search depths play hundreds of games against each other

**How to run:** tap **Runtime → Run all**. It needs no files and takes about a minute.

## 1. Write and compile the C++ engine

In [ ]:
%%writefile connect4.cpp
// Connect 4 AI engine (C++17): bitboards + negamax with alpha-beta pruning.
//
// Build:  g++ -O2 -std=c++17 connect4.cpp -o c4
//
// Usage:
//   ./c4 bench [maxDepth]             node counts and speed from the empty board (CSV)
//   ./c4 play <depthA> <depthB> <N>   AI vs AI tournament, one CSV row per game
//   ./c4 best <depth> <moves>         best move after a move string, e.g. "3344"
//   ./c4 compare [maxDepth]           nodes searched WITH vs WITHOUT alpha-beta pruning (CSV)
//
// Board layout: bit index = column * 7 + row (row 0 is the bottom).
// Row 6 of each column is an empty "sentinel" bit so shifts never wrap between columns.

#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <random>
#include <string>
#include <vector>

using namespace std;
typedef uint64_t u64;
typedef chrono::steady_clock Clock;

static const int COLS = 7, ROWS = 6;
static const int WIN = 100000, INF = 1000000000;
static const int ORDER[COLS] = {3, 2, 4, 1, 5, 0, 6};   // centre columns first = better pruning
static const int SC_ME[5]  = {0, 0, 3, 12, 0};          // my pieces in an open window of 4
static const int SC_OPP[5] = {0, 0, 3, 14, 0};          // opponent pieces in an open window
static const u64 CENTER = 0x3FULL << 21;                 // the 6 cells of column 3

static vector<u64> WINDOWS;    // all 69 possible four-in-a-row lines as bitmasks
static long long nodes = 0;    // positions searched (for benchmarking)
static bool PRUNE = true;      // false = plain minimax (used only to measure what pruning saves)

static void initWindows() {
    const int dirs[4][2] = {{1, 0}, {0, 1}, {1, 1}, {1, -1}};
    for (int c = 0; c < COLS; c++)
        for (int r = 0; r < ROWS; r++)
            for (auto& d : dirs) {
                int ec = c + 3 * d[0], er = r + 3 * d[1];
                if (ec < 0 || ec >= COLS || er < 0 || er >= ROWS) continue;
                u64 mask = 0;
                for (int k = 0; k < 4; k++) mask |= 1ULL << ((c + k * d[0]) * 7 + (r + k * d[1]));
                WINDOWS.push_back(mask);
            }
}

struct Pos {
    u64 b[2];       // b[0] = first player's stones, b[1] = second player's
    int h[COLS];    // bit index of the next free cell in each column
    int moves;
    Pos() : moves(0) {
        b[0] = b[1] = 0;
        for (int c = 0; c < COLS; c++) h[c] = c * 7;
    }
    bool can(int c) const { return h[c] < c * 7 + ROWS; }
    void play(int c) { b[moves & 1] |= 1ULL << h[c]; h[c]++; moves++; }
    void undo(int c) { moves--; h[c]--; b[moves & 1] ^= 1ULL << h[c]; }
};

// Four in a row? Check vertical, horizontal and both diagonals with shifts.
static inline bool isWin(u64 bb) {
    static const int dirs[4] = {1, 7, 6, 8};
    for (int d : dirs) {
        u64 t = bb & (bb >> d);
        if (t & (t >> (2 * d))) return true;
    }
    return false;
}

// Heuristic score from the point of view of the player to move.
static int evaluate(const Pos& p) {
    int me = p.moves & 1;
    u64 A = p.b[me], B = p.b[me ^ 1];
    int s = 0;
    for (u64 w : WINDOWS) {
        int m = __builtin_popcountll(A & w), o = __builtin_popcountll(B & w);
        if (o == 0) s += SC_ME[m];
        else if (m == 0) s -= SC_OPP[o];
    }
    s += 3 * (__builtin_popcountll(A & CENTER) - __builtin_popcountll(B & CENTER));
    return s;
}

// Negamax with alpha-beta pruning. Returns the score for the player to move.
static int negamax(Pos& p, int depth, int alpha, int beta) {
    nodes++;
    if (p.moves == COLS * ROWS) return 0;                       // draw
    int me = p.moves & 1;
    for (int i = 0; i < COLS; i++) {                            // can I win right now?
        int c = ORDER[i];
        if (p.can(c) && isWin(p.b[me] | (1ULL << p.h[c]))) return WIN - p.moves;
    }
    if (depth == 0) return evaluate(p);
    int best = -INF;
    for (int i = 0; i < COLS; i++) {
        int c = ORDER[i];
        if (!p.can(c)) continue;
        p.play(c);
        int v = -negamax(p, depth - 1, -beta, -alpha);
        p.undo(c);
        if (v > best) best = v;
        if (v > alpha) alpha = v;
        if (PRUNE && alpha >= beta) break;                      // prune
    }
    return best;
}

struct Result { int col; int score; long long nodes; };

static Result bestMove(Pos& p, int depth) {
    nodes = 0;
    int me = p.moves & 1;
    for (int i = 0; i < COLS; i++) {
        int c = ORDER[i];
        if (p.can(c) && isWin(p.b[me] | (1ULL << p.h[c])))
            return {c, WIN - p.moves, 1};
    }
    int alpha = -INF, bestC = -1;
    for (int i = 0; i < COLS; i++) {
        int c = ORDER[i];
        if (!p.can(c)) continue;
        p.play(c);
        int v = -negamax(p, depth - 1, -INF, -alpha);
        p.undo(c);
        if (bestC < 0 || v > alpha) { alpha = v; bestC = c; }
    }
    return {bestC, alpha, nodes};
}

static double msSince(Clock::time_point t0) {
    return chrono::duration<double, milli>(Clock::now() - t0).count();
}

static void runBench(int maxDepth) {
    printf("depth,best_col,score,nodes,ms,nodes_per_sec\n");
    for (int d = 1; d <= maxDepth; d++) {
        Pos p;
        auto t0 = Clock::now();
        Result r = bestMove(p, d);
        double ms = msSince(t0);
        printf("%d,%d,%d,%lld,%.3f,%.0f\n", d, r.col, r.score, r.nodes, ms, ms > 0 ? r.nodes / ms * 1000.0 : 0.0);
        fflush(stdout);
    }
}

static void runCompare(int maxDepth) {
    printf("depth,nodes_pruned,nodes_full,ms_pruned,ms_full,saved_pct\n");
    for (int d = 1; d <= maxDepth; d++) {
        Pos p; PRUNE = true;
        auto t0 = Clock::now(); Result a = bestMove(p, d); double msA = msSince(t0);
        Pos q; PRUNE = false;
        t0 = Clock::now(); Result b = bestMove(q, d); double msB = msSince(t0);
        PRUNE = true;
        printf("%d,%lld,%lld,%.3f,%.3f,%.2f\n", d, a.nodes, b.nodes, msA, msB, 100.0 * (1.0 - (double)a.nodes / (double)b.nodes));
        if (a.col != b.col || a.score != b.score) fprintf(stderr, "MISMATCH at depth %d\n", d);   // pruning must never change the answer
        fflush(stdout);
    }
}

static void runPlay(int dA, int dB, int games) {
    printf("game,a_first,winner,moves,a_moves,b_moves,a_ms,b_ms,a_nodes,b_nodes\n");
    for (int g = 0; g < games; g++) {
        mt19937 rng(1000 + g);
        Pos p;
        bool aFirst = (g % 2 == 0);
        for (int i = 0; i < 2; i++) {                           // 2 random opening moves for variety
            int c;
            do { c = rng() % COLS; } while (!p.can(c));
            p.play(c);
        }
        double ms[2] = {0, 0};
        long long nd[2] = {0, 0};
        int cnt[2] = {0, 0};
        char winner = 'D';
        while (true) {
            int side = p.moves & 1;
            bool isA = (side == 0) == aFirst;
            int who = isA ? 0 : 1;
            auto t0 = Clock::now();
            Result r = bestMove(p, isA ? dA : dB);
            ms[who] += msSince(t0);
            nd[who] += r.nodes;
            cnt[who]++;
            p.play(r.col);
            if (isWin(p.b[side])) { winner = isA ? 'A' : 'B'; break; }
            if (p.moves == COLS * ROWS) break;
        }
        printf("%d,%d,%c,%d,%d,%d,%.3f,%.3f,%lld,%lld\n", g, aFirst ? 1 : 0, winner, p.moves,
               cnt[0], cnt[1], ms[0], ms[1], nd[0], nd[1]);
        fflush(stdout);
    }
}

static void runBest(int depth, const string& seq) {
    Pos p;
    for (char ch : seq) {
        int c = ch - '0';
        if (c < 0 || c >= COLS || !p.can(c)) { fprintf(stderr, "bad move string\n"); exit(1); }
        p.play(c);
    }
    Result r = bestMove(p, depth);
    printf("%d %d %lld\n", r.col, r.score, r.nodes);
}

int main(int argc, char** argv) {
    initWindows();
    if (argc >= 2 && !strcmp(argv[1], "bench")) {
        runBench(argc >= 3 ? atoi(argv[2]) : 8);
    } else if (argc >= 2 && !strcmp(argv[1], "compare")) {
        runCompare(argc >= 3 ? atoi(argv[2]) : 8);
    } else if (argc >= 5 && !strcmp(argv[1], "play")) {
        runPlay(atoi(argv[2]), atoi(argv[3]), atoi(argv[4]));
    } else if (argc >= 4 && !strcmp(argv[1], "best")) {
        runBest(atoi(argv[2]), argv[3]);
    } else {
        printf("Connect 4 AI engine\n"
               "  %s bench [maxDepth]\n"
               "  %s play <depthA> <depthB> <games>\n"
               "  %s best <depth> <moves>\n"
               "  %s compare [maxDepth]\n", argv[0], argv[0], argv[0], argv[0]);
    }
    return 0;
}

In [ ]:
import subprocess

build = subprocess.run(["g++", "-O2", "-std=c++17", "connect4.cpp", "-o", "c4"], capture_output=True, text=True)
print("Compiled OK" if build.returncode == 0 else build.stderr)

## 2. What does alpha-beta pruning save?

In [ ]:
import io, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def run(*args):
    out = subprocess.run(["./c4", *map(str, args)], capture_output=True, text=True, check=True)
    return out.stdout

cmp_df = pd.read_csv(io.StringIO(run("compare", 8)))
print(cmp_df.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(cmp_df)); w = 0.4
ax[0].bar(x - w/2, cmp_df.nodes_full, w, label="Plain minimax", color="#d9534f")
ax[0].bar(x + w/2, cmp_df.nodes_pruned, w, label="Alpha-beta", color="#2e8b57")
ax[0].set_yscale("log"); ax[0].set_xticks(x); ax[0].set_xticklabels(cmp_df.depth)
ax[0].set_xlabel("Search depth (moves ahead)"); ax[0].set_ylabel("Positions searched (log scale)")
ax[0].set_title("Positions searched from the empty board"); ax[0].legend()
ax[1].plot(cmp_df.depth, cmp_df.saved_pct, marker="o", color="#2e8b57")
ax[1].set_ylim(0, 100); ax[1].set_xlabel("Search depth"); ax[1].set_ylabel("% of positions skipped")
ax[1].set_title("Work saved by pruning"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

last = cmp_df.iloc[-1]
print(f"At depth {int(last.depth)}: {int(last.nodes_full):,} positions without pruning vs {int(last.nodes_pruned):,} with pruning "
      f"({last.saved_pct:.2f}% skipped), and the chosen move is identical.")

## 3. Search speed by depth

In [ ]:
bench = pd.read_csv(io.StringIO(run("bench", 10)))
print(bench.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(bench.depth, bench.nodes, marker="o"); ax[0].set_yscale("log")
ax[0].set_xlabel("Search depth"); ax[0].set_ylabel("Positions searched (log)"); ax[0].set_title("Search size grows with depth"); ax[0].grid(alpha=.3)
ax[1].plot(bench.depth, bench.nodes_per_sec / 1e6, marker="o", color="#6f42c1")
ax[1].set_xlabel("Search depth"); ax[1].set_ylabel("Million positions per second"); ax[1].set_title("Engine speed"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. Tournament: does searching deeper win more?

Every pair of depths (1 to 7) plays 30 games. Each game starts with 2 random moves for variety,
and the two sides take turns going first, so the match-up is fair.

In [ ]:
DEPTHS = [1, 2, 3, 4, 5, 6, 7]
GAMES = 30

frames = []
for a, b in itertools.combinations(DEPTHS, 2):
    df = pd.read_csv(io.StringIO(run("play", a, b, GAMES)))
    df["depth_a"], df["depth_b"] = a, b
    frames.append(df)
games = pd.concat(frames, ignore_index=True)
print(f"{len(games)} games played")
games.head()

In [ ]:
# Score matrix: row depth vs column depth (win = 1, draw = 0.5, loss = 0)
n = len(DEPTHS)
M = np.full((n, n), np.nan)
for a, b in itertools.combinations(DEPTHS, 2):
    g = games[(games.depth_a == a) & (games.depth_b == b)]
    score_a = ((g.winner == "A").sum() + 0.5 * (g.winner == "D").sum()) / len(g)
    M[DEPTHS.index(a), DEPTHS.index(b)] = score_a
    M[DEPTHS.index(b), DEPTHS.index(a)] = 1 - score_a

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(DEPTHS); ax.set_yticks(range(n)); ax.set_yticklabels(DEPTHS)
ax.set_xlabel("Opponent depth"); ax.set_ylabel("Player depth"); ax.set_title("Score of row player vs column player")
for i in range(n):
    for j in range(n):
        if i != j: ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, label="Score (1 = always wins)"); plt.tight_layout(); plt.show()

In [ ]:
# How much does each extra move of lookahead help the deeper player?
rows = []
for _, r in games.iterrows():
    deeper = "B" if r.depth_b > r.depth_a else "A"
    gap = abs(r.depth_b - r.depth_a)
    res = "win" if r.winner == deeper else ("draw" if r.winner == "D" else "loss")
    rows.append((gap, res))
gap_df = pd.DataFrame(rows, columns=["gap", "result"])
table = pd.crosstab(gap_df.gap, gap_df.result, normalize="index").reindex(columns=["win", "draw", "loss"], fill_value=0) * 100
print(table.round(1).to_string())

table.plot(kind="bar", stacked=True, figsize=(7, 4), color=["#2e8b57", "#bbbbbb", "#d9534f"])
plt.xlabel("Depth advantage of the deeper player"); plt.ylabel("% of games"); plt.title("Deeper search wins more often")
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0)); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Cost of thinking: time and positions per move at each depth
recs = []
for _, r in games.iterrows():
    recs.append((r.depth_a, r.a_ms / r.a_moves, r.a_nodes / r.a_moves))
    recs.append((r.depth_b, r.b_ms / r.b_moves, r.b_nodes / r.b_moves))
cost = pd.DataFrame(recs, columns=["depth", "ms_per_move", "nodes_per_move"]).groupby("depth").mean()
print(cost.round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(cost.index, cost.ms_per_move, marker="o", color="#6f42c1")
ax.set_yscale("log"); ax.set_xlabel("Search depth"); ax.set_ylabel("Average ms per move (log)"); ax.set_title("Thinking time per move"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Plain-English summary computed from the results above
deep, shallow = max(DEPTHS), min(DEPTHS)
print(f"Games played: {len(games)}")
print(f"Depth {deep} vs depth {shallow}: depth {deep} scored {M[DEPTHS.index(deep), DEPTHS.index(shallow)]:.0%}")
top = table.loc[table.index.max()]
print(f"With a {int(table.index.max())}-move depth advantage the deeper player won {top['win']:.0f}% of games.")
print(f"Alpha-beta pruning skipped {last.saved_pct:.2f}% of positions at depth {int(last.depth)} without changing the move.")